# Patrones avanzados de recursión en UnifyWeaver

Este cuaderno de trabajo demuestra los cuatro patrones principales de recursión que UnifyWeaver puede detectar y optimizar:

1. **Recursión de cola** - Bucles iterativos con acumuladores
2. **Recursión lineal** - Llamada recursiva única con memoización
3. **Recursión de árbol** - Múltiples llamadas recursivas en partes estructurales
4. **Recursión mutua** - Predicados que se llaman entre sí en ciclos

## Objetivos de aprendizaje

- Comprender los diferentes patrones de recursión
- Ver cómo UnifyWeaver detecta y optimiza cada patrón
- Comparar características de rendimiento
- Aprender cuándo usar cada patrón

## Configuración

Inicializa el entorno de UnifyWeaver.

In [ ]:
% Cargar inicialización
['../init'].

% Cargar los módulos necesarios
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Patrón 1: Recursión de cola

La recursión de cola utiliza un acumulador para transportar resultados intermedios, y la llamada recursiva es la **última acción** en la función.

### Ejemplo: Contar elementos en una lista

In [ ]:
% Definir count_items con recursión de cola
:- dynamic count_items/3.

% Caso base: lista vacía, retornar el acumulador
count_items([], Acc, Acc).

% Caso recursivo: incrementar el acumulador, aplicar recursión sobre la cola
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← ¡Posición de cola!

### Probar en Prolog

In [ ]:
% Prueba: contar elementos en [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### Comprobar la detección de patrones

In [ ]:
% Comprobar si se detecta como recursión de cola
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Compilar a Bash

In [ ]:
% Compilar y guardar
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### Probar el Bash generado

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Contando elementos en [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## Patrón 2: Recursión lineal

La recursión lineal tiene **exactamente una** llamada recursiva por cláusula, con computación que ocurre después de que la llamada recursiva retorna.

### Ejemplo: Factorial

In [ ]:
% Definir factorial
:- dynamic factorial/2.

% Caso base
factorial(0, 1).

% Caso recursivo: exactamente UNA llamada recursiva
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← Una llamada recursiva
    F is N * F1.        % ← Cálculo después de la llamada

### Probar en Prolog

In [ ]:
% Prueba: factorial de 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### Comprobar la detección de patrones

In [ ]:
% Comprobar si se detecta como recursión lineal
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Compilar a Bash

In [ ]:
% Compilar y guardar
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Mantener solo las definiciones de funciones; Brush trata los scripts cargados con source como ejecución directa
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### Probar el Bash generado

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial de 5:"
factorial 5 ""
echo ""
echo "Factorial de 10:"
factorial 10 ""

## Patrón 3: Recursión de árbol

La recursión de árbol realiza **múltiples** llamadas recursivas para procesar diferentes partes de una estructura.

### Ejemplo: Suma de árbol

In [ ]:
% Definir tree_sum para árboles binarios
% Formato de árbol: [Valor, SubárbolIzquierdo, SubárbolDerecho] o []
:- dynamic tree_sum/2.

% Caso base: el árbol vacío tiene suma 0
tree_sum([], 0).

% Caso recursivo: suma = valor + suma_izquierda + suma_derecha
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← Primera llamada recursiva
    tree_sum(R, RS),   % ← Segunda llamada recursiva
    Sum is V + LS + RS.

### Probar en Prolog

In [ ]:
% Prueba: tree_sum de [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Compilar a Bash

In [ ]:
% Compilar y guardar
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### Probar el Bash generado

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Suma del árbol [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## Patrón 4: Recursión mutua

La recursión mutua ocurre cuando dos o más predicados se llaman entre sí en un ciclo.

### Ejemplo: Par e impar

In [ ]:
% Definir is_even e is_odd con recursión mutua
:- dynamic is_even/1.
:- dynamic is_odd/1.

% Caso base de is_even
is_even(0).

% Recursivo de is_even: N es par si N-1 es impar
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Llama a is_odd

% Caso base de is_odd
is_odd(1).

% Recursivo de is_odd: N es impar si N-1 es par
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Llama a is_even

### Probar en Prolog

In [ ]:
% Probar par/impar
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### Comprobar la recursión mutua

In [ ]:
% Construir grafo de llamadas y encontrar SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Compilar a Bash

In [ ]:
% Compilar el grupo de recursión mutua
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### Probar el Bash generado

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Probando is_even e is_odd:"
is_even 0 >/dev/null && echo "✓ 0 es par"
is_even 4 >/dev/null && echo "✓ 4 es par"
is_odd 3 >/dev/null && echo "✓ 3 es impar"
is_odd 7 >/dev/null && echo "✓ 7 es impar"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 no es par"

## Comparación de patrones

Comparemos las características de cada patrón:

| Patrón | Llamadas recursivas | Optimización | Complejidad espacial | Más adecuado para |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **De cola** | 1 (posición de cola) | Bucle iterativo | O(1) | Acumuladores, barridos lineales |
| **Lineal** | 1 (cualquier posición) | Plegado (fold) + memoización | O(n) tabla de memoización | Fibonacci, factorial |
| **De árbol** | 2+ (partes estructurales) | Descomposición estructural | O(profundidad) pila | Operaciones en árboles/grafos |
| **Mutua** | 1+ (entre predicados) | Memoización compartida | O(n) tabla compartida | Par/impar, definiciones mutuas |

## Orden de detección de patrones

UnifyWeaver prueba los patrones en este orden:

1. **Recursión de cola** (más eficiente)
2. **Recursión lineal** (a menos que esté prohibida)
3. **Recursión de árbol** (estructural)
4. **Recursión mutua** (detección de SCC)
5. **Recursión básica** (reserva / fallback)

Es posible influir en la detección con `forbid_linear_recursion/1`.

## Ejercicio: ¡Tu turno!

Prueba definir y compilar estos predicados:

### 1. Suma con recursión de cola
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. Fibonacci con recursión lineal
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. Altura de un árbol
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% ¡Tu código aquí!


## Resumen

En este cuaderno de trabajo, aprendiste:

✅ Los cuatro patrones principales de recursión en UnifyWeaver

✅ Cómo definir cada patrón en Prolog

✅ Cómo detecta y optimiza UnifyWeaver cada patrón

✅ Las características de rendimiento de cada patrón

✅ Cuándo usar cada patrón

## Próximos pasos

¡Continúa con el **Cuaderno de trabajo 3: Visualización de grafos de llamadas** para aprender sobre análisis avanzado de código y visualización!